# 05 — Modeling

Train and compare models on both feature sets (cheap vs. full) to answer 
the real question from feature engineering: how much predictive power do 
the expensive, self-collected signals actually add over cheap, easily 
available metadata?

Starting with Logistic Regression as an interpretable baseline, then 
Random Forest for comparison. Evaluated with precision/recall/F1 rather 
than raw accuracy, since risk_category is imbalanced (~50% Low, ~33% High, 
~17% Medium) - a model could hit ~50% accuracy just by always predicting 
"Low" and that would be a useless model, not a good one.

In [6]:
import joblib
import pandas as pd
import numpy as np

data = joblib.load('../data/model_ready_data.pkl')
X_cheap_train = data['X_cheap_train']
X_cheap_test = data['X_cheap_test']
X_full_train = data['X_full_train']
X_full_test = data['X_full_test']
y_train = data['y_train']
y_test = data['y_test']

print("Cheap train/test:", X_cheap_train.shape, X_cheap_test.shape)
print("Full train/test:", X_full_train.shape, X_full_test.shape)

Cheap train/test: (398, 24) (100, 24)
Full train/test: (398, 34) (100, 34)


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

scaler_cheap = StandardScaler()
X_cheap_train_scaled = scaler_cheap.fit_transform(X_cheap_train)
X_cheap_test_scaled = scaler_cheap.transform(X_cheap_test)

logreg_cheap = LogisticRegression(max_iter=1000, random_state=42)
logreg_cheap.fit(X_cheap_train_scaled, y_train)

y_pred_cheap = logreg_cheap.predict(X_cheap_test_scaled)

print("=== Logistic Regression - CHEAP features ===")
print(classification_report(y_test, y_pred_cheap))

=== Logistic Regression - CHEAP features ===
              precision    recall  f1-score   support

        High       0.66      0.82      0.73        33
         Low       0.78      0.80      0.79        50
      Medium       0.62      0.29      0.40        17

    accuracy                           0.72       100
   macro avg       0.69      0.64      0.64       100
weighted avg       0.72      0.72      0.70       100



In [8]:
scaler_full = StandardScaler()
X_full_train_scaled = scaler_full.fit_transform(X_full_train)
X_full_test_scaled = scaler_full.transform(X_full_test)

logreg_full = LogisticRegression(max_iter=1000, random_state=42)
logreg_full.fit(X_full_train_scaled, y_train)

y_pred_full = logreg_full.predict(X_full_test_scaled)

print("=== Logistic Regression - FULL features ===")
print(classification_report(y_test, y_pred_full))

=== Logistic Regression - FULL features ===
              precision    recall  f1-score   support

        High       0.63      0.82      0.71        33
         Low       0.81      0.78      0.80        50
      Medium       0.56      0.29      0.38        17

    accuracy                           0.71       100
   macro avg       0.67      0.63      0.63       100
weighted avg       0.71      0.71      0.70       100



In [9]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest doesn't need feature scaling - trees split on raw thresholds
rf_cheap = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf_cheap.fit(X_cheap_train, y_train)
y_pred_rf_cheap = rf_cheap.predict(X_cheap_test)

rf_full = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf_full.fit(X_full_train, y_train)
y_pred_rf_full = rf_full.predict(X_full_test)

print("=== Random Forest - CHEAP features ===")
print(classification_report(y_test, y_pred_rf_cheap))
print("\n=== Random Forest - FULL features ===")
print(classification_report(y_test, y_pred_rf_full))

=== Random Forest - CHEAP features ===
              precision    recall  f1-score   support

        High       0.66      0.88      0.75        33
         Low       0.82      0.74      0.78        50
      Medium       0.45      0.29      0.36        17

    accuracy                           0.71       100
   macro avg       0.65      0.64      0.63       100
weighted avg       0.71      0.71      0.70       100


=== Random Forest - FULL features ===
              precision    recall  f1-score   support

        High       0.68      0.79      0.73        33
         Low       0.84      0.74      0.79        50
      Medium       0.44      0.47      0.46        17

    accuracy                           0.71       100
   macro avg       0.66      0.67      0.66       100
weighted avg       0.72      0.71      0.71       100



In [5]:
importances = pd.Series(rf_full.feature_importances_, index=X_full_train.columns).sort_values(ascending=False)
importances.head(10)

commits_per_month_365d            0.206163
commits_per_month_90d             0.204821
merged_prs_total                  0.065885
total_mentionable_users           0.056921
closed_issues_total               0.052126
forks                             0.050173
age_days                          0.049579
stars                             0.044679
distinct_recent_commit_authors    0.041323
pr_merge_rate_recent              0.033740
dtype: float64

In [10]:
from sklearn.metrics import f1_score

comparison = pd.DataFrame({
    'Cheap (F1)': [
        f1_score(y_test, y_pred_rf_cheap, labels=['High'], average='macro'),
        f1_score(y_test, y_pred_rf_cheap, labels=['Low'], average='macro'),
        f1_score(y_test, y_pred_rf_cheap, labels=['Medium'], average='macro'),
    ],
    'Full (F1)': [
        f1_score(y_test, y_pred_rf_full, labels=['High'], average='macro'),
        f1_score(y_test, y_pred_rf_full, labels=['Low'], average='macro'),
        f1_score(y_test, y_pred_rf_full, labels=['Medium'], average='macro'),
    ],
}, index=['High', 'Low', 'Medium'])
comparison['Improvement'] = comparison['Full (F1)'] - comparison['Cheap (F1)']
comparison

,Cheap (F1),Full (F1),Improvement
High,0.753247,0.732394,-0.020852
Low,0.778947,0.787234,0.008287
Medium,0.357143,0.457143,0.100000


In [11]:
importances_fixed = pd.Series(rf_full.feature_importances_, index=X_full_train.columns).sort_values(ascending=False)
importances_fixed.head(10)

merged_prs_total               0.119176
closed_issues_total            0.109394
age_days                       0.101169
total_mentionable_users        0.084648
stars                          0.080543
forks                          0.079974
pr_merge_rate_recent           0.061635
open_issues_total              0.058064
open_prs_total                 0.050346
median_pr_merge_days_recent    0.049535
dtype: float64

## Modeling Summary

| Model | Features | Accuracy | Medium F1 |
|---|---|---|---|
| Logistic Regression | Cheap | 0.72 | 0.40 |
| Logistic Regression | Full | 0.71 | 0.43 |
| Random Forest | Cheap | 0.71 | 0.36 |
| Random Forest | Full | 0.71 | 0.46 |

**Data leakage caught and fixed:** an initial version of the "full" feature 
set included `commits_per_month_90d`, `commits_per_month_365d`, and 
`distinct_recent_commit_authors` - three features that were direct inputs 
to the `risk_score` formula itself. This produced a suspiciously perfect 
1.00 F1 on the Low class and 92% accuracy; feature importance confirmed 
these three features accounted for ~45% of the model's decisions. Removed 
them and re-ran - honest full-feature performance is much more modest, 
as expected.

**Key finding:** overall accuracy is nearly identical between cheap and 
full feature sets (~71-72%), meaning headline accuracy alone would suggest 
the expensive data collection wasn't worth it. But per-class breakdown 
tells a different story: full features meaningfully improve Medium-risk 
detection (F1 +0.10, recall 0.29 → 0.47) at a small cost to High-risk 
recall. The deep signals specifically help with the *ambiguous middle 
ground* - repos that are neither clearly thriving nor clearly dead - 
which is arguably the most valuable case to get right in a real triage tool.

**Feature importance (full model):** dominated by cumulative activity 
volume (`merged_prs_total`, `closed_issues_total`) rather than the 
maintenance-quality signals (PR merge rate, issue resolution speed) 
that were expected to matter most - a genuine "expected X, found Y" 
result worth investigating further in a v2.

**Next:** explainability (SHAP) on individual predictions, then a 
short business-recommendations writeup.

In [12]:
import joblib

joblib.dump({
    'rf_cheap': rf_cheap,
    'rf_full': rf_full,
}, '../models/random_forest_models.pkl')

print("Models saved.")

Models saved.
